# 9 Editing in the terminal: Vim

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part III — From commands to scripts</span>
    <span class="bp-meta">Notebook&nbsp;9</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    The first notebook where you <em>write</em> rather than only read: the modal
    editor that lives on every cluster, from survival to search-and-replace.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v1.0.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root and source the validation gate. data/ is
# read-only; everything we edit is a fresh scratch copy. Vim is interactive, so
# the rendered cells below produce their results with *scripted* Vim (vim -es) —
# genuine Vim, run non-interactively, exactly reproducible.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"

## What this notebook is about

Part II was all about *reading*: pulling things out of read-only data. Part III
is where you start *writing*: scripts, configs, submission files of your own. And
writing needs an editor.

The editor is **Vim**. Two reasons it is worth the climb, said plainly. First, it
is *everywhere*: every Linux box and every cluster has it, often as the only
editor installed. Second, you will be dropped into it whether you ask or not
(`git` opens it for commit messages, `crontab -e` opens it, countless tools default
to it), so "I'll just avoid Vim" is not actually on the menu. Better to learn it
properly once.

As always: the files you edit here are scripts and configs, **just text.** No
physics required.

## How to practise this notebook

Here is the one unusual thing about this notebook. Vim is *interactive* (it takes
over your whole terminal and responds to keystrokes), so it **cannot run in the
grey cells** on this page. There is no live keyboard here.

So the workspace for this notebook is the **live terminal**, not the page:

```{admonition} Practise this notebook in your terminal
:class: tip
**[▶ Open the live terminal](https://mybinder.org/v2/gh/ramador09/bash-primer-public/HEAD?urlpath=shell/)**
and keep it beside this page. Every "try it" below is something you *do* there. The
cells on this page show the *before and after* of an edit (produced with scripted
Vim so they are reproducible), but the editing itself is yours to do live.
```

## A. The modal model

The single idea that unlocks Vim (and the single thing that traps every beginner
who skips it) is that Vim is **modal**. The keys do different things depending on
which *mode* you are in. There are four worth knowing:

- **Normal**: where you start. Letters are **commands**, not text (`d` deletes,
  `i` starts inserting). This is home base.
- **Insert**: where you actually type text, like a normal editor.
- **Visual**: where you select a region, then act on it.
- **Command-line**: where you type `:` commands like `:w` (save) and `:q` (quit).

`Esc` always takes you back to Normal. When in doubt, press `Esc` and you are home.

<div style="background:#1b2233;border:1px solid #0e1422;border-radius:8px;padding:8px 20px 16px;margin:18px 0;">
  <div style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:.12em;text-transform:uppercase;color:#c0851a;font-weight:600;margin:8px 0 6px;">Vim's modes, and how to move between them</div>
  <pre style="background:transparent;color:#cdd2db;margin:0;font-family:'JetBrains Mono',monospace;font-size:0.84rem;line-height:1.5;">              ┌──────────────────────────────────────────────┐
              │                 NORMAL  mode                 │
              │      you start here — letters are commands    │
              └────┬─────────────────┬────────────────┬──────┘
              press i a o      press v V Ctrl-v     press :
                   │                 │                │
                   ▼                 ▼                ▼
                INSERT            VISUAL         COMMAND-LINE
              (type text)    (select a region)  ( :w  :q  :wq  :%s/… )
                   │                 │                │
                   └─────── Esc ─────┴──── Esc ───────┘  ──▶ back to NORMAL</pre>
</div>

`vim` is the program you launch; here is its card. Everything *inside* Vim (the
modes, motions, and `:` commands) lives in the reference tables below, not in the
card.

```{command-card} vim
```

## B. Survival first

Before anything else, the four things that stop you ever being trapped. The most
common question about Vim is "how do I get *out* of it", so here is the answer,
up front.

<div class="bp-card">
  <span class="bp-card-cmd">Survival kit</span> — <span class="bp-card-job">the keys that open, save, and quit. Press <code>Esc</code> first if unsure where you are.</span>
  <table>
    <tr><td>vim&nbsp;file</td><td>open (or create) the file</td></tr>
    <tr><td>i</td><td>enter Insert mode and start typing</td></tr>
    <tr><td>Esc</td><td>leave Insert, back to Normal</td></tr>
    <tr><td>:w</td><td>write (save)</td></tr>
    <tr><td>:q</td><td>quit</td></tr>
    <tr><td>:wq&nbsp;&nbsp;or&nbsp;ZZ</td><td>save <b>and</b> quit</td></tr>
    <tr><td>:q!</td><td>quit and <b>throw away</b> unsaved changes: the escape hatch</td></tr>
  </table>
</div>

That last one, `:q!`, is your panic button: it always gets you out, no matter what
mess you have made, without saving it. Commit it to memory now.

```{admonition} Try it (in your terminal)
:class: tip
`vim scratch-hello.txt`, press `i`, type a line, press `Esc`, type `:wq`, Enter.
You just created and saved a file in Vim. Now `vim scratch-hello.txt` again to
reopen it, and `:q` to leave.
```

## C. Moving around

In Normal mode you move the cursor without arrow keys (your hands never leave the
home row). Motions also combine with **counts**: a number before a motion repeats
it, so `5j` moves down five lines.

<div class="bp-card">
  <span class="bp-card-cmd">Motions</span> — <span class="bp-card-job">move the cursor in Normal mode (no arrow keys needed).</span>
  <table>
    <tr><td>h&nbsp;j&nbsp;k&nbsp;l</td><td>left, down, up, right (one character/line)</td></tr>
    <tr><td>w&nbsp;&nbsp;b&nbsp;&nbsp;e</td><td>forward a word, back a word, to word-end</td></tr>
    <tr><td>0&nbsp;&nbsp;^&nbsp;&nbsp;$</td><td>start of line, first non-blank, end of line</td></tr>
    <tr><td>gg&nbsp;&nbsp;G</td><td>top of file, bottom of file</td></tr>
    <tr><td>:N</td><td>jump to line number N</td></tr>
    <tr><td>5j&nbsp;&nbsp;3w</td><td>counts: repeat a motion (down 5 lines; forward 3 words)</td></tr>
  </table>
</div>

## D. Editing — verbs and motions

The part that makes Vim worth learning rhymes with something you
already know. In Part II you saw the Unix idea: small tools **composed** with
pipes. Vim has the same idea for editing: small **verbs** composed with **motions**.

A verb is an operator: `d` delete, `c` change, `y` yank (copy). A motion says
*how far*. Snap them together and you get a precise edit:

- `dw`: **d**elete a **w**ord; `d$`: delete to end of line; `3dw`: delete three words.
- `cw`: **c**hange a word (delete it and drop straight into Insert mode).
- `y$`: yank to end of line; `p`: paste it back.

Once you see the grammar, you do not memorise hundreds of commands; you combine a
handful of verbs with the motions from the table above.

<div class="bp-card">
  <span class="bp-card-cmd">Verbs &amp; everyday edits</span> — <span class="bp-card-job">operators compose with the motions above; these are the ones you reach for daily.</span>
  <table>
    <tr><td>x</td><td>delete the character under the cursor</td></tr>
    <tr><td>dd&nbsp;&nbsp;yy</td><td>delete / yank (copy) the whole line</td></tr>
    <tr><td>dw&nbsp;&nbsp;cw</td><td>delete / change a word (verb + motion)</td></tr>
    <tr><td>p&nbsp;&nbsp;P</td><td>paste after / before the cursor</td></tr>
    <tr><td>o&nbsp;&nbsp;O</td><td>open a new line below / above and start typing</td></tr>
    <tr><td>u&nbsp;&nbsp;Ctrl-r</td><td>undo / redo</td></tr>
    <tr><td>.</td><td>repeat the last change (astonishingly useful)</td></tr>
  </table>
</div>

## E. Visual mode — select first, then operate

Verbs-and-motions act *forward* from the cursor. Visual mode is the other way
round: **select a region first, watch it highlight, then apply one verb to all of
it.** There are three flavours, and the third is a Vim signature.

- **`v`** is charwise: select character by character.
- **`V`** is linewise: select whole lines.
- **`Ctrl-v`** is **blockwise**: select a *rectangular column*. This is the one
  nothing else does as cleanly, and the one you will reach for constantly on
  structured text: scripts, config files, aligned columns in an input deck.

Once a region is selected, operate on it: `d` delete, `y` yank, `c` change, `>` /
`<` indent, `~` toggle case.

### The block-mode power moves

Blockwise visual mode earns its own paragraph because of what it does to *columns*.
Mark a rectangle with `Ctrl-v`, then:

- **`I`**: insert text at the start of **every** selected line. Type it once, press
  `Esc`, and it appears on all of them.
- **`A`**: the same, but append (use `$` first to catch ragged line-ends).
- **`d`** deletes the column, **c** changes it, **`r`** replaces it.

The everyday payoff is **block-commenting**: put `# ` in front of a run of lines in
one move: `Ctrl-v`, select down with `j`, `I`, type `# `, `Esc`. Here is that
edit's before and after, produced on a scratch file with scripted Vim so you can
see the result (in your terminal you would do it with the keystrokes just
described):

In [2]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'set cutoff 300\nset maxiter 50\nset window 4\n' > scratch/deck.txt

In [3]:
cat scratch/deck.txt

set cutoff 300


set maxiter 50


set window 4


In [4]:
vim -es -c '%s/^/# /' -c 'wq' scratch/deck.txt

In [5]:
cat scratch/deck.txt

# set cutoff 300


# set maxiter 50


# set window 4


Every line now carries a leading `# `: three lines commented in one gesture. The
reverse (deleting a leading column with `Ctrl-v` then `d`) is just as quick.

<div class="bp-card">
  <span class="bp-card-cmd">Visual mode</span> — <span class="bp-card-job">enter a mode, extend with motions, then operate. <code>Esc</code> applies block edits to every line.</span>
  <table>
    <tr><td>v&nbsp;&nbsp;V</td><td>charwise / linewise selection</td></tr>
    <tr><td>Ctrl-v</td><td>blockwise (rectangular column) selection</td></tr>
    <tr><td>d&nbsp;&nbsp;y&nbsp;&nbsp;c</td><td>delete / yank / change the selection</td></tr>
    <tr><td>&gt;&nbsp;&nbsp;&lt;&nbsp;&nbsp;~</td><td>indent right / left; toggle case</td></tr>
    <tr><td>Ctrl-v&nbsp;…&nbsp;I</td><td>block: insert on every selected line (then <code>Esc</code>)</td></tr>
    <tr><td>Ctrl-v&nbsp;…&nbsp;A</td><td>block: append on every line (<code>$</code> first for ragged ends)</td></tr>
  </table>
</div>

## F. Search

Finding things is the same `/` you already met in `less` and `man` (they page
through `less`, remember). In Normal mode:

- **`/pattern`** then Enter searches forward; **`?pattern`** searches backward.
- **`n`** jumps to the next match; **`N`** to the previous one.
- **`*`** searches for the word currently under the cursor (no typing needed).

## G. Search and replace — the power tool

This is the move that justifies the whole notebook, and it carries a gift from
Part II: **the regular expressions you learned for `grep` and `sed` are close
cousins of the ones Vim searches with** — close, but not identical. In Vim's
default *magic* level the characters `+ ? | ( )` are **literal**, so an
Extended pattern like `warning|error` has to be written `warning\|error`, and
`[0-9]+` becomes `[0-9]\+`. The shortcut is `\v` ("very magic") at the front of
the pattern, which makes Vim read the rest exactly as `grep -E` would:
`:%s/\v(warning|error)/ALERT/g`. The command lives in command-line mode:

- `:s/old/new/`: replace the first `old` on the **current line**.
- `:s/old/new/g`: replace **g**lobally on the current line (every match).
- `:%s/old/new/g`: `%` means "all lines", so this replaces **every** occurrence in
  the file. This is the one you will use most.
- `:10,20s/old/new/g`: only within a line range.
- `:%s/old/new/gc`: add `c` to **confirm** each change (`y`/`n`) before it happens.

Here it is on a scratch parameter file. Before:

In [6]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'cutoff   = 300\nmaxsteps = 300\nwindow   = 300\n' > scratch/params.txt

In [7]:
cat scratch/params.txt

cutoff   = 300


maxsteps = 300


window   = 300


In your terminal you would open the file and type `:%s/300/500/g`. Run
non-interactively for the page, that is:

In [8]:
vim -es -c '%s/300/500/g' -c 'wq' scratch/params.txt

In [9]:
cat scratch/params.txt

cutoff   = 500


maxsteps = 500


window   = 500


Every `300` became `500`, across all three lines, in one command. That is the
edit you will run a thousand times: change a parameter everywhere, rename a
variable throughout a script, retarget a path across a config.

## H. A word on `nano` (and why Vim anyway)

If Vim's modes are genuinely not for you today, there is a gentler editor: **`nano`**.
It is *modeless* (you just type, like a text box) and it shows its key shortcuts
along the bottom of the screen (`^O` to save, `^X` to exit, where `^` means Ctrl).
It is a perfectly fine fallback, and on your own machine you may prefer it.

```{command-card} nano
```

But the reason this notebook leads with Vim and not `nano` is simple: when you
`ssh` into a cluster at 2 a.m. to fix a submission script, Vim is what is *there*,
and the tool that drops you in unannounced is Vim, not `nano`. Learning it is an
investment that pays off on every machine you will ever touch.

```{admonition} Optional: a friendlier Vim
:class: note
Vim is endlessly configurable through a `~/.vimrc` file. A few beginner-friendly
lines make it much more pleasant: `set number` (show line numbers), `syntax on`
(colour), `set incsearch` (search as you type). You do not need this
to start; just know the knob exists for when you are ready.
```

## Exercises

These are different from the earlier notebooks': the editing happens **in your
terminal**, live, so
[**open it now**](https://mybinder.org/v2/gh/ramador09/bash-primer-public/HEAD?urlpath=shell/)
if it is not already beside you. Each exercise tells you what to do in Vim; the
graded ✓ checks the file you end up with. (On this page the answer is produced with scripted Vim so
the build stays green, but the real practice is yours to do in the terminal.) As
always, we work on fresh `scratch/` copies, never on `data/`.

### Warm-up — Survival drill

In your terminal: `vim scratch/hello.txt`, press `i`, type `Hello, Vim`, press
`Esc`, save and quit with `:wq`. Reopen it to confirm it stuck.

In [10]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [11]:
# (solution hidden on the public site)


Hello, Vim


In [12]:
check '[ "$(cat scratch/hello.txt)" = "Hello, Vim" ]' \
      "the file holds the line you inserted"

✓ the file holds the line you inserted


### Applied 1 — Fix a broken file

This scratch script has a typo: `eccho` should be `echo`. Open it in Vim, fix the
typo (with `cw`, or `x`, or `:s`, your choice), save, and the script will run.
It is just text; you do not need to know what it computes.

In [13]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf '#!/usr/bin/env bash\neccho "binding energy: 42 meV"\n' > scratch/run.sh

In [14]:
# (solution hidden on the public site)


binding energy: 42 meV


In [15]:
check 'bash scratch/run.sh 2>/dev/null | grep -q "binding energy: 42 meV"' \
      "the typo is fixed and the script now runs"

✓ the typo is fixed and the script now runs


### Applied 2 — Search and replace (the centrepiece)

This config sets the value `300` on every line. Open it in Vim and change **every**
`300` to `500` in one command with `:%s/300/500/g`. None should be missed.

In [16]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'cutoff   = 300\nmaxsteps = 300\nwindow   = 300\nseed     = 300\n' > scratch/config.txt

In [17]:
# (solution hidden on the public site)


cutoff   = 500


maxsteps = 500


window   = 500


seed     = 500


In [18]:
check '! grep -q 300 scratch/config.txt && [ "$(grep -c 500 scratch/config.txt)" -eq 4 ]' \
      "every 300 became 500 — all four, none missed"

✓ every 300 became 500 — all four, none missed


### Applied 3 — Block edit a column (visual block)

The real payoff of block mode. This scratch file has a run of lines. **(a)**
Block-comment all of them: `Ctrl-v`, select down with `j`, `I`, type `# `, `Esc`.
**(b)** On the second file, delete the leading `| ` column: `Ctrl-v`, select the
two-character column down, `d`.

In [19]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'echo alpha\necho beta\necho gamma\n' > scratch/lines.sh
printf '| 1.0 | x |\n| 2.0 | y |\n| 3.0 | z |\n' > scratch/table.txt

In [20]:
# (solution hidden on the public site)


# echo alpha


# echo beta


# echo gamma


1.0 | x |


2.0 | y |


3.0 | z |


In [21]:
check '[ "$(grep -c "^# echo" scratch/lines.sh)" -eq 3 ] && ! grep -q "^| " scratch/table.txt' \
      "every line was commented, and the leading column was removed"

✓ every line was commented, and the leading column was removed


### Composite — putting it together (author from scratch)

Open a brand-new file `scratch/greet.sh` in Vim and write a small script by hand:
a shebang line, then two `echo` lines. Save it, then run it. This exercises
everything (open, insert, new lines with `o`, save) and points straight at Notebook
12, where you will write real scripts.

In [22]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [23]:
# (solution hidden on the public site)


#!/usr/bin/env bash


echo "written in Vim"


echo "the date is $(date +%F)"


written in Vim


the date is 2026-08-09


In [24]:
check '[ -f scratch/greet.sh ] && head -n 1 scratch/greet.sh | grep -q "env bash" && bash scratch/greet.sh | grep -q "written in Vim"' \
      "the file exists, starts with a shebang, and runs"

✓ the file exists, starts with a shebang, and runs


### Optional stretch (terminal-only) — Efficiency challenge

No ✓: do this one for speed, in your terminal. Take any of the scratch files above
and redo an edit using **counts, verbs, and motions** rather than retyping: delete
three words with `3dw`, change a word with `cw`, copy a line with `yy` and paste it
with `p`, then undo it all with `u`. Then do a one-line `:%s/.../.../` that would
have taken many manual edits. Feeling *where* the leverage is (a few keystrokes
standing in for a lot of typing) is the whole point of learning Vim.

## Outlook

You can now create and change text in the terminal: the missing piece before you
write anything of your own. Next (Notebook 10): **quoting and expansion**, the
rules that decide how the shell reads what you type. They are quietly responsible
for most of the subtle bugs people hit once they start writing real commands and
scripts, so they are worth getting straight before you write much more.

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practise&rdquo; box above to do
    every edit yourself; nothing to install. The published notebooks ship
    <b>without worked solutions</b>; if you would like the reference solutions
    (to teach from or to check your own work), get in touch:
    <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>